# Experiment 4: a systems view

**Questions.**

1. Conway's B3/S23 is one of $2^{17}$ life-like rules without B0. Sampling rules at random and classifying their behaviour automatically: how common is behaviour like Life's, and where does it sit?
2. Is Life robust? What happens to the 17 rules that differ from it by one digit?

The causal loop diagrams that summarise experiments 1–3 are in the [write-up](../docs/experiments/04-systems.md).

| Term | Meaning |
|---|---|
| dead / frozen / periodic | most soups die out / settle into still patterns / settle into short cycles |
| chaotic | most soups never settle, and one flipped cell spreads until the runs are unrelated |
| complex | long transients before settling, or no settling without full decorrelation (Life-like) |
| Langton's $\lambda$ | share of the 18 birth/survival entries that produce a live cell: $(\lvert B\rvert + \lvert S\rvert)/18$; Life has $\lambda = 3/18 \approx 0.17$ |

## Predictions (written before any run)

| # | Question | Prediction |
|---|---|---|
| 1 | Most common class among random rules | dead or frozen |
| 2 | Share of complex rules | 5–10 % |
| 3 | Do chaotic rules have high or low $\lambda$? Do complex rules cluster at an "edge of chaos"? | high; yes |
| 4 | How many of Life's 17 one-digit neighbours are still complex? | about 3 |
| 5 | Which leads to chaos more: extra birth digits or extra survival digits? | extra survival digits |
| 6 | (Causal loops) On the lattice, does a balancing loop (freezing) or a reinforcing loop (spreading) dominate Life in the long run? | balancing (freezing) |

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from life.rules import CONWAY, parse_rule
from life.rulespace import (
    CLASSES,
    classify_rule,
    langton_lambda,
    neighbours,
    rule_features,
    sample_rules,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = ROOT / "docs" / "figures"
DATA = ROOT / "data" / "exp04"
FIGURES.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

COLORS = {
    "dead": "#9aa3ab",
    "frozen": "#3b6ea8",
    "periodic": "#6aa84f",
    "complex": "#e69f00",
    "chaotic": "#d1495b",
}
FIELDS = ["settled", "transient", "period", "density", "activity", "damage"]

## 4b. A random sample of rule space

500 distinct rules drawn uniformly (B0 excluded). Each rule runs 10 random soups (density 0.35) on a 32 × 32 torus for up to 2000 generations; features and classes are defined in `life.rulespace` and [ADR 0004](../docs/adr/0004-rule-classification.md).

In [ ]:
N_RULES = 500
rules = sample_rules(N_RULES, np.random.default_rng(2026))

features = []
start = time.perf_counter()
for i, rule in enumerate(rules):
    features.append(rule_features(rule, seed=i))
    if (i + 1) % 100 == 0:
        print(f"{i + 1} rules in {time.perf_counter() - start:.0f} s")

lam = np.array([langton_lambda(r) for r in rules])
classes = np.array([classify_rule(f) for f in features])
table = np.array([[getattr(f, k) for k in FIELDS] for f in features])
with open(DATA / "rule_scan.csv", "w", encoding="utf-8", newline="\n") as fh:
    fh.write("rule,lambda,class," + ",".join(FIELDS) + "\n")
    for rule, lam_i, c, row in zip(rules, lam, classes, table, strict=True):
        fh.write(f"{rule},{lam_i:.4f},{c}," + ",".join(f"{v:.6g}" for v in row) + "\n")

In [ ]:
print(f"{'class':>9} {'rules':>6} {'share':>6}  examples")
for c in CLASSES:
    idx = np.flatnonzero(classes == c)
    examples = ", ".join(str(rules[i]) for i in idx[:3])
    print(f"{c:>9} {len(idx):>6} {len(idx) / N_RULES:>6.1%}  {examples}")
print()
life_class = classify_rule(rule_features(CONWAY))
print(f"Life (B3/S23) under the same settings: {life_class}")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
counts = [np.sum(classes == c) for c in CLASSES]
left.bar(CLASSES, counts, color=[COLORS[c] for c in CLASSES])
left.set_ylabel("rules")
left.set_title(f"Classes of {N_RULES} random life-like rules")

digits = np.round(lam * 18).astype(int)
levels = np.arange(digits.min(), digits.max() + 1)
bottom = np.zeros(len(levels))
for c in CLASSES:
    share = np.array(
        [np.mean(classes[digits == k] == c) if np.any(digits == k) else 0 for k in levels]
    )
    right.bar(levels / 18, share, bottom=bottom, width=1 / 18 * 0.9, color=COLORS[c], label=c)
    bottom += share
right.axvline(3 / 18, color="black", ls="--", lw=1)
right.text(3 / 18, 1.02, "Life", ha="center", fontsize=8)
right.set_xlabel("Langton's λ = (|B| + |S|) / 18")
right.set_ylabel("share of rules")
right.set_title("Class mix along λ")
right.legend(fontsize=8, loc="center right")
fig.tight_layout()
fig.savefig(FIGURES / "exp04_rulespace.png", dpi=150)
plt.show()

### Birth digits or survival digits?

In [ ]:
n_birth = np.array([len(r.birth) for r in rules])
n_survive = np.array([len(r.survive) for r in rules])

fig, ax = plt.subplots(figsize=(7, 3.6))
for counts, label, marker in [
    (n_birth, "number of birth digits |B|", "o-"),
    (n_survive, "number of survival digits |S|", "s-"),
]:
    ks = np.arange(counts.min(), counts.max() + 1)
    share = [
        np.mean(classes[counts == k] == "chaotic") if np.sum(counts == k) >= 5 else np.nan
        for k in ks
    ]
    ax.plot(ks, share, marker, label=label)
ax.set_xlabel("number of digits")
ax.set_ylabel("share of chaotic rules")
ax.set_title("Which digits push a rule into chaos?")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "exp04_birth_survive.png", dpi=150)
plt.show()

for name, counts in [("birth", n_birth), ("survive", n_survive)]:
    r = np.corrcoef(counts, classes == "chaotic")[0, 1]
    print(f"correlation of the number of {name} digits with being chaotic: {r:+.2f}")
print(
    "chaotic rules containing B1:",
    np.mean([1 in rules[i].birth for i in np.flatnonzero(classes == "chaotic")]),
)
print("all rules containing B1:   ", np.mean([1 in r.birth for r in rules]))

### How much do the conclusions depend on the thresholds?

Both thresholds of `classify_rule` (100 generations for a "long" transient, half of full decorrelation for "chaotic") are halved and increased by half.

In [ ]:
print(f"{'long transient':>15} {'chaotic damage':>15}  " + " ".join(f"{c:>9}" for c in CLASSES))
for lt in [50, 100, 150]:
    for cd in [0.25, 0.5, 0.75]:
        alt = np.array([classify_rule(f, long_transient=lt, chaotic_damage=cd) for f in features])
        shares = " ".join(f"{np.mean(alt == c):>9.1%}" for c in CLASSES)
        print(f"{lt:>15} {cd:>15}  {shares}")

## 4c. Life's neighbourhood

The 17 rules that differ from B3/S23 by adding or removing one digit (B0 excluded), classified with the same settings.

In [ ]:
near = neighbours(CONWAY)
near_features = [rule_features(r, seed=1000 + i) for i, r in enumerate(near)]
near_classes = [classify_rule(f) for f in near_features]

print(
    f"{'rule':>16} {'change':>8} {'class':>9} {'settled':>8} {'transient':>10} {'period':>7} "
    f"{'density':>8} {'damage':>7}"
)
for r, f, c in zip(near, near_features, near_classes, strict=True):
    if r.birth != CONWAY.birth:
        k = next(iter(r.birth ^ CONWAY.birth))
        change = f"{'+' if k in r.birth else '-'}B{k}"
    else:
        k = next(iter(r.survive ^ CONWAY.survive))
        change = f"{'+' if k in r.survive else '-'}S{k}"
    print(
        f"{str(r):>16} {change:>8} {c:>9} {f.settled:>8.0%} {f.transient:>10.0f} {f.period:>7.0f} "
        f"{f.density:>8.3f} {f.damage:>7.2f}"
    )
print()
for c in CLASSES:
    print(f"{c:>9}: {near_classes.count(c)} of 17")

with open(DATA / "life_neighbours.csv", "w", encoding="utf-8", newline="\n") as fh:
    fh.write("rule,class," + ",".join(FIELDS) + "\n")
    for r, f, c in zip(near, near_features, near_classes, strict=True):
        fh.write(f"{r},{c}," + ",".join(f"{getattr(f, k):.6g}" for k in FIELDS) + "\n")

### Check on a larger grid

Classes depend on the grid size through the transient threshold. Life, its complex neighbours and a few reference rules are re-run on a 64 × 64 torus (5 soups each, cap 5000), with the transient threshold scaled up by the same factor that Life's own transient grows.

In [ ]:
check = [CONWAY, parse_rule("B2/S"), parse_rule("B3/S012345678")]
check += [r for r, c in zip(near, near_classes, strict=True) if c == "complex"]
life_small = rule_features(CONWAY).transient
life_large = rule_features(CONWAY, size=64, runs=5, max_steps=5000).transient
scale = life_large / life_small
print(
    f"Life's median transient: {life_small:.0f} on 32 × 32, {life_large:.0f} on 64 × 64 "
    f"(threshold scaled by {scale:.1f})"
)
for r in check:
    small = classify_rule(rule_features(r))
    f = rule_features(r, size=64, runs=5, max_steps=5000)
    large = classify_rule(f, long_transient=100 * scale)
    print(
        f"{str(r):>16}: 32 × 32 {small:>9}   64 × 64 {large:>9}   "
        f"(transient {f.transient:.0f}, settled {f.settled:.0%})"
    )

## Results vs. predictions

*To be written after looking at the output above.*